In [593]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from DateTimeEncoder.DateEncoder import DateEncoder
from DateTimeEncoder.DurationEncoder import DurationEncoder
from DateTimeEncoder.TimeEncoder import TimeEncoder
from sklearn.preprocessing import FunctionTransformer
import numpy as np


from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

In [594]:
df = pd.read_csv("flights.csv")

In [595]:
print(df.columns)

Index(['index', 'Airline', 'Date_of_Journey', 'Source', 'Destination',
       'Dep_Time', 'Duration', 'Total_Stops', 'Additional_Info', 'Price'],
      dtype='str')


In [596]:
# Combine Source and Destination into a single Route feature
df["Route"] = df["Source"].astype(str) + "_" + df["Destination"].astype(str)

In [597]:
categorical_cols = ["Airline", "Additional_Info"]
destination_cols = ["Source", "Destination"]
duration_cols = ["Duration"]
time_cols = ["Dep_Time"]
stops_order = [["non-stop", "1 stop", "2 stops", "3 stops", "4 stops"]]
date_cols = ["Date_of_Journey"]

In [598]:
# Duplicate Categories in Additional_Info
df["Additional_Info"] = df["Additional_Info"].replace("No info", "No Info")
# Count occurrences of each category
info_counts = df["Additional_Info"].value_counts()

# Identify categories that appear infrequently (e.g., fewer than 10 times)
rare_categories = info_counts[info_counts < 10].index

# 4. Replace rare categories with 'Other'
df["Additional_Info"] = df["Additional_Info"].replace(rare_categories, "Other")

In [599]:
X = df.drop(columns=["Price","index"])
y = df["Price"]
X.columns

Index(['Airline', 'Date_of_Journey', 'Source', 'Destination', 'Dep_Time',
       'Duration', 'Total_Stops', 'Additional_Info', 'Route'],
      dtype='str')

In [600]:
def check_is_weekend(X):
    # Converts dayofweek (0-6) to 1 if Friday (4), Saturday (5), or Sunday (6), else 0
    return (X >= 4).astype(int)

def weekend_feature_names(transformer, input_features):
    return ["Date_of_Journey_is_weekend"]


# Pipeline step for indicating weekends
is_weekend_transformer = Pipeline(
    steps=[
        ("date_enc", DateEncoder(features=["dayofweek"])),
        (
            "to_weekend",
            FunctionTransformer(
                check_is_weekend, feature_names_out=weekend_feature_names
            ),
        ),
    ])


In [601]:
#Prepocess tranform
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(
                handle_unknown="ignore",
                feature_name_combiner=lambda feature, category: f"{category}",
            ),
            categorical_cols,
        ),
        (
            "destination",
            OneHotEncoder(
                handle_unknown="ignore",
                feature_name_combiner=lambda feature, category: f"{feature}_{category}",
            ),
            destination_cols,
        ),
        (
            "date",
            DateEncoder(features=["month", "day", "dayofweek"]),
            date_cols,
        ),
        ("weekend", is_weekend_transformer, date_cols),
        ("time", TimeEncoder(features=["hour", "minute"]), time_cols),
        ("duration", DurationEncoder(), duration_cols),
        (
            "stops",
            OrdinalEncoder(
                categories=stops_order,
                handle_unknown="use_encoded_value",
                unknown_value=-1,
            ),
            ["Total_Stops"],
        ),
    ],
    remainder="drop",
    sparse_threshold=0,
    verbose_feature_names_out=False,
)

In [602]:
processed_matrixs=preprocessor.fit_transform(X)

In [603]:
cols = preprocessor.get_feature_names_out()
num_columns = 3

for i in range(0, len(cols), num_columns):
    row = cols[i : i + num_columns]
    print("".join(f"{item:<35}" for item in row))

Air Asia                           Air India                          GoAir                              
IndiGo                             Jet Airways                        Jet Airways Business               
Multiple carriers                  Multiple carriers Premium economy  SpiceJet                           
Trujet                             Vistara                            Vistara Premium economy            
1 Long layover                     In-flight meal not included        No Info                            
No check-in baggage included       Other                              Source_Banglore                    
Source_Chennai                     Source_Delhi                       Source_Kolkata                     
Source_Mumbai                      Destination_Banglore               Destination_Cochin                 
Destination_Delhi                  Destination_Hyderabad              Destination_Kolkata                
Destination_New Delhi              Date_of_Jou

In [604]:
X_train, X_valid, y_train, y_valid = train_test_split(X, y, train_size=0.8, test_size=0.2, random_state=0)

In [611]:
#Defining the model
model = RandomForestRegressor(n_estimators=300, max_depth=25)

In [612]:
pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", model)
])

In [ ]:
pipeline.fit(X_train, y_train)

In [ ]:
pre_vals = pipeline.predict(X_valid)

In [ ]:
mean_absolute_error(y_valid, pre_vals)